# Track Limits Computer Vision Training (YOLOv8)
This notebook is designed to be run in **Google Colab** with a **T4 GPU**.

### Dataset Augmentation Notes (Roboflow)
As discussed, since track limits are enforced by physical track boundaries, you should configure the following **Basic Augmentations** in Roboflow before generating the dataset version:
- **Rotation**: Small-angle (e.g. ±15°) to simulate camera tilt/pan.
- **Crop**: To simulate zoom changes and framing.
- **Brightness / Exposure**: To handle different race times, shadows, and weather.
- **Blur / Noise / Motion Blur**: Critical for fast-panning F1 broadcast cameras.
- **Camera Gain / Hue / Saturation**: To mimic different broadcast grading and lighting.
*Note: Skip vertical flips and 90° rotations, as they create physically impossible camera angles.*

In [ ]:
!nvidia-smi

## 1. Install Dependencies

In [ ]:
!pip install -q ultralytics roboflow

## 2. Download Dataset from Roboflow
Replace `YOUR_API_KEY`, `WORKSPACE`, `PROJECT`, and `VERSION` with your actual Roboflow details.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_API_KEY_HERE")
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT_NAME")
version = project.version(1)
dataset = version.download("yolov8")

## 3. Train YOLOv8 Model
Training the model on the downloaded dataset. The T4 GPU should handle this efficiently.

In [ ]:
from ultralytics import YOLO

# Load a YOLOv8 Nano model (recommended for real-time video inference)
# You can upgrade to yolov8s.pt or yolov8m.pt if you need higher accuracy and have the compute.
model = YOLO("yolov8n-seg.pt")  # using -seg for segmentation, use yolov8n.pt for bounding boxes

# Train the model
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0, # Use the T4 GPU
    project="track_limits",
    name="run_1"
)

## 4. Evaluate and Export
Check the validation metrics and export the weights so you can download them and use them in your local pipeline.

In [ ]:
# Display confusion matrix and validation results
import glob
from IPython.display import Image, display

for image_path in glob.glob("track_limits/run_1/val*.jpg"):
    display(Image(filename=image_path, width=600))

In [ ]:
from google.colab import files

# The best weights are saved in track_limits/run_1/weights/best.pt
weights_path = "track_limits/run_1/weights/best.pt"

# Download the weights to your local machine
files.download(weights_path)